In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
pyspark.__file__

'/Users/carloskim/Developer/DE_zoomcamp/DE_zoomcamp/06_homework/.venv/lib/python3.13/site-packages/pyspark/__init__.py'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("06_homework") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/06 12:27:02 WARN Utils: Your hostname, Carloss-MacBook-Pro-8.local, resolves to a loopback address: 127.0.0.1; using 192.168.2.28 instead (on interface en0)
26/03/06 12:27:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 12:27:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-06 11:32:24--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.83, 18.239.38.181, 18.239.38.163, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  10.9MB/s    in 6.3s    

2026-03-06 11:32:30 (10.7 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [5]:
!wc yellow_tripdata_2025-11.parquet

  282291 1939741 71134255 yellow_tripdata_2025-11.parquet


In [6]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [7]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [10]:
df.head(5)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 49, 7), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 1, 1, 22), passenger_count=1, trip_distance=2.28, RatecodeID=1, store_and_fwd_flag='N', PULocationID=142, DOLocationID=237, payment_type=1, fare_amount=14.2, extra=1.0, mta_tax=0.5, tip_amount=4.99, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=24.94, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2025, 11, 1,

In [11]:
df_repartitioned = df.repartition(4)

In [12]:
df_repartitioned.write.parquet("yellow_tripdata_2025-11")

In [13]:
df = spark.read.parquet('yellow_tripdata_2025-11/')

In [15]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [17]:
from pyspark.sql.functions import col

In [18]:
df.filter(
    (col("tpep_pickup_datetime") >= "2025-11-15") &
    (col("tpep_pickup_datetime") < "2025-11-16")
).count()

162604

In [19]:
df.createOrReplaceTempView("trips")

In [24]:
spark.sql("""
SELECT 
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0) AS longest_trip_hours
FROM trips
""").show()

[Stage 12:=============================>                            (4 + 4) / 8]

+------------------+
|longest_trip_hours|
+------------------+
|         90.646667|
+------------------+



In [25]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-06 12:20:38--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.83, 18.239.38.163, 18.239.38.147, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-06 12:20:38 (13.9 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [26]:
zone_df = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

zone_df.createOrReplaceTempView("zones")

In [27]:
spark.sql("""
SELECT
    z.Zone,
    COUNT(*) AS pickup_count
FROM trips t
JOIN zones z
ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY pickup_count ASC
LIMIT 1
""").show()

[Stage 17:=============================>                            (4 + 4) / 8]

+-------------+------------+
|         Zone|pickup_count|
+-------------+------------+
|Arden Heights|           1|
+-------------+------------+



In [4]:
spark.stop()